# 🚀 Building a Simple LLM with Google Colab — A–Z Practical

**Level:** Beginner / Diploma / Higher Diploma  
**Platform:** Google Colab  
**Language:** Python  
**Main libraries:** PyTorch, Transformers, Hugging Face

## Learning goals

By the end of this practical, students should be able to:

- Explain what an LLM is.
- Explain tokens, vocabulary, training, loss, parameters and inference.
- Build a tiny next-word language model.
- Use a pretrained language model in Google Colab.
- Generate text from prompts.
- Understand the difference between **training an LLM** and **using a pretrained LLM**.
- Build a simple terminal-style AI chatbot.

> **Important:** We are not training a ChatGPT-sized model from scratch. We first build a tiny educational language model and then use a real pretrained model.


## 🧠 Part 1 — What is an LLM?

**LLM = Large Language Model**

An LLM is a machine-learning model trained on a large amount of text so that it can learn patterns in language and generate text.

A simplified flow is:

```text
Text Dataset
     ↓
Tokenization
     ↓
Numbers / Tokens
     ↓
Neural Network
     ↓
Training
     ↓
Learned Parameters
     ↓
Language Model
     ↓
Prompt → Next-token prediction → Generated text
```

### Example

Prompt:

```text
Artificial intelligence is
```

The model may predict:

```text
powerful
```

Then it continues predicting more tokens.

### Key idea

An LLM repeatedly predicts what token is likely to come next.


## 🤖 AI → ML → Deep Learning → NLP → LLM

A useful hierarchy for students:

```text
Artificial Intelligence
        ↓
Machine Learning
        ↓
Deep Learning
        ↓
Natural Language Processing
        ↓
Large Language Models
```

These areas overlap, but they are not exactly the same thing.

### Important terms

| Term | Simple meaning |
|---|---|
| AI | Machines performing tasks associated with intelligence |
| ML | Learning patterns from data |
| Deep Learning | ML using neural networks with many layers |
| NLP | Working with human language |
| LLM | Large neural language model trained on huge text datasets |
| Token | A small unit of text processed by a model |
| Parameter | A learned numerical value inside a model |
| Prompt | Input given to a model |
| Inference | Using a trained model to produce an output |
| Fine-tuning | Further training a pretrained model on a specific dataset |


# Part 2 — Create a Google Colab Notebook

1. Open Google Colab.
2. Create a **New Notebook**.
3. Rename it:

```text
Simple_LLM_AZ_Practical
```

4. For this notebook, a GPU is useful but not required for the first exercises.

### Enable GPU

Go to:

**Runtime → Change runtime type → Hardware accelerator → GPU**

Then run the next cell.


In [ ]:
# Check Python and GPU information

import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. CPU mode will be used.")


# Part 3 — Create Our Tiny Dataset

For learning purposes, we will create a very small text dataset.

A real LLM uses enormously larger datasets. This tiny dataset is only for demonstrating the basic learning idea.


In [ ]:
text = """
Python is a programming language.
Python is easy to learn.
Python is useful for software development.
Python is useful for artificial intelligence.
Artificial intelligence can learn from data.
Machine learning is a part of artificial intelligence.
Deep learning uses neural networks.
Large language models can generate text.
Language models learn patterns from text.
Software engineering is the process of designing and building software.
Programming helps us solve problems.
"""

print(text)


## 🔤 Tokenization

A model cannot directly process ordinary sentences as text.

We need to convert text into smaller units called **tokens**.

For our first demonstration, we will use words as tokens.

Real LLM tokenizers are more sophisticated and may split words into subwords.


In [ ]:
# Simple word tokenization

words = text.lower().split()

print("First 40 tokens:")
print(words[:40])

print("\nTotal tokens:", len(words))


## 📚 Create a Vocabulary

A vocabulary contains the unique tokens known by our model.


In [ ]:
# Create vocabulary

vocab = sorted(set(words))

print("Vocabulary size:", len(vocab))
print("\nVocabulary:")
print(vocab)


## 🔢 Convert Words to Numbers

Neural networks work with numbers.

We therefore create two mappings:

```text
word → number
number → word
```


In [ ]:
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

print("Word to ID:")
print(word_to_id)

print("\nExample:")
print("python →", word_to_id["python"])


# Part 4 — Create Training Examples

Our tiny model will learn a very simple task:

> Given one word, predict the next word.

For example:

```text
python → is
is → a
a → programming
programming → language.
```

This is a simplified version of the next-token prediction idea used in language models.


In [ ]:
# Create input → target pairs

inputs = []
targets = []

for i in range(len(words) - 1):
    inputs.append(word_to_id[words[i]])
    targets.append(word_to_id[words[i + 1]])

print("First 20 training examples:")

for i in range(20):
    print(
        id_to_word[inputs[i]],
        "→",
        id_to_word[targets[i]]
    )


# Part 5 — Build a Tiny Neural Language Model

Now we will use PyTorch.

Our simplified model has:

```text
Input word ID
     ↓
Embedding
     ↓
Linear layer
     ↓
Vocabulary scores
     ↓
Predicted next word
```

### Embedding

An embedding converts a token ID into a vector of numbers.

### Linear layer

The linear layer produces a score for every possible next word.


In [ ]:
import torch
import torch.nn as nn

# Convert our training data into tensors
X = torch.tensor(inputs, dtype=torch.long)
y = torch.tensor(targets, dtype=torch.long)

print("Input shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
class SimpleLanguageModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim=32):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.linear = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, x):
        x = self.embedding(x)
        x = self.linear(x)
        return x


model = SimpleLanguageModel(len(vocab))

print(model)


# Part 6 — Loss Function and Optimizer

During training:

```text
Input
  ↓
Model prediction
  ↓
Compare with correct answer
  ↓
Calculate loss
  ↓
Backpropagation
  ↓
Update parameters
  ↓
Repeat
```

**Loss** tells us how wrong the prediction is.

The optimizer changes the model's parameters to reduce the loss.


In [ ]:
loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

print("Loss function and optimizer created.")


# Part 7 — Train the Tiny Model

We will train for several epochs.

An **epoch** means one training pass through the training data.


In [ ]:
epochs = 1000

for epoch in range(epochs):

    # Reset old gradients
    optimizer.zero_grad()

    # Forward pass
    output = model(X)

    # Calculate loss
    loss = loss_function(output, y)

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:4d} | Loss: {loss.item():.4f}"
        )


## 📉 Understanding the Loss

If training is working, the loss should generally decrease.

For example:

```text
Epoch 0     Loss: high
Epoch 100   Loss: lower
Epoch 500   Loss: much lower
Epoch 900   Loss: lower
```

The exact values depend on the dataset and random initialization.


# Part 8 — Make a Prediction

Now the model can predict a next word.

We will use `softmax` to convert scores into probabilities.


In [ ]:
def predict_next_word(word):

    word = word.lower()

    if word not in word_to_id:
        return "Unknown word"

    input_id = torch.tensor(
        [word_to_id[word]],
        dtype=torch.long
    )

    with torch.no_grad():
        output = model(input_id)

    probabilities = torch.softmax(output, dim=-1)

    next_id = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    return id_to_word[next_id]


print("python →", predict_next_word("python"))
print("programming →", predict_next_word("programming"))
print("artificial →", predict_next_word("artificial"))


# Part 9 — Generate Multiple Words

One prediction is not very exciting.

Let's repeatedly predict the next word.


In [ ]:
def generate_text(start_word, number_of_words=10):

    current_word = start_word.lower()
    result = [current_word]

    for _ in range(number_of_words):

        next_word = predict_next_word(current_word)

        if next_word == "Unknown word":
            break

        result.append(next_word)
        current_word = next_word

    return " ".join(result)


print(generate_text("python", 10))
print(generate_text("artificial", 10))


## ⚠️ Why does our tiny model produce strange text?

This is expected.

Our model has:

- a tiny dataset
- a tiny vocabulary
- very few parameters
- a very simple architecture
- no Transformer architecture
- no large-scale training

Therefore it is **not comparable to modern LLMs**.

But the important learning concept is demonstrated:

```text
Text
 ↓
Tokens
 ↓
Numbers
 ↓
Neural Network
 ↓
Training
 ↓
Next-token prediction
 ↓
Generated text
```


# Part 10 — What is a Real LLM?

A modern LLM is much more complex.

A simplified architecture looks like:

```text
Huge Text Dataset
       ↓
    Tokenizer
       ↓
      Tokens
       ↓
   Embeddings
       ↓
 Transformer Layers
       ↓
 Learned Parameters
       ↓
   Output Probabilities
       ↓
   Next Token
```

Modern LLMs use Transformer-based architectures and very large numbers of learned parameters.

### Important distinction

**Training an LLM from scratch**

```text
Huge dataset
+ Huge compute
+ Large model
+ Long training
= Very expensive
```

**Using a pretrained model**

```text
Pretrained model
       ↓
Download/load
       ↓
Run inference
       ↓
Generate text
```

For students and small projects, the second approach is much more practical.


# Part 11 — Install Hugging Face Transformers

Now we will use a real pretrained language model.

Hugging Face provides tools and model repositories that make it easier to work with pretrained models.


In [ ]:
!pip -q install transformers


# Part 12 — Load a Pretrained Model

We will use **GPT-2** as a teaching example.

GPT-2 is an older, relatively small generative language model compared with current frontier models, which makes it useful for demonstrations.

The model was trained before this notebook and we are using it for inference.


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2"
)

print("GPT-2 text generation pipeline is ready.")


# Part 13 — Generate Text

Let's give the model a prompt.


In [ ]:
prompt = "Artificial intelligence is"

result = generator(
    prompt,
    max_new_tokens=40,
    num_return_sequences=1
)

print(result[0]["generated_text"])


## 🎯 Experiment with Different Prompts

Try changing the prompt.

Examples:

```text
Python is useful because
```

```text
Software engineering is
```

```text
A good student should
```

```text
The future of artificial intelligence is
```

Remember: generated output can vary and may contain incorrect or nonsensical information.


In [ ]:
prompt = "Python is useful because"

result = generator(
    prompt,
    max_new_tokens=50,
    num_return_sequences=1
)

print(result[0]["generated_text"])


# Part 14 — Understanding Prompt and Completion

The basic generation flow is:

```text
User Prompt
     ↓
Tokenizer
     ↓
Token IDs
     ↓
Pretrained Model
     ↓
Probability Distribution
     ↓
Next Token
     ↓
More tokens
     ↓
Generated Text
```

This is called **inference**.

We are not training GPT-2 in this section.


# Part 15 — Build a Simple Chatbot

Now we will create a very simple command-line chatbot.

This is an educational demonstration, not a production chatbot.


In [ ]:
def ask_llm(question):
    result = generator(
        question,
        max_new_tokens=60,
        num_return_sequences=1,
        do_sample=True
    )

    return result[0]["generated_text"]


print("AI:", ask_llm("Explain programming in simple English."))


## 💬 Interactive Chatbot

Run the following cell and type questions.

Press **Enter** on an empty line to stop.


In [ ]:
print("🤖 Simple LLM Chatbot")
print("Type an empty line to exit.\n")

while True:

    user_input = input("You: ")

    if user_input.strip() == "":
        print("Goodbye!")
        break

    response = ask_llm(user_input)

    print("AI:", response)
    print()


# Part 16 — Important: This is not ChatGPT

Our chatbot is very basic.

A production LLM application normally needs much more:

```text
User Interface
      ↓
Backend
      ↓
Prompt Management
      ↓
LLM
      ↓
Safety / Validation
      ↓
Response
```

Modern AI applications may also use:

- RAG
- embeddings
- vector databases
- tool calling
- agents
- structured output
- monitoring
- evaluation
- authentication


# Part 17 — Fine-Tuning Concept

Suppose we want an AI assistant specialized for a college.

We could prepare examples such as:

```text
Question:
What is the purpose of an API?

Answer:
An API allows software systems to communicate...
```

```text
Question:
What is REST?

Answer:
REST is an architectural style for designing networked applications...
```

A simplified fine-tuning workflow is:

```text
Pretrained Model
       ↓
Domain Dataset
       ↓
Fine-Tuning
       ↓
Specialized Model
```

### Important

Fine-tuning is different from simply giving a prompt.

**Prompting:**

```text
Pretrained model + instruction
```

**Fine-tuning:**

```text
Pretrained model + training examples
          ↓
Updated model parameters
```


# Part 18 — LLM vs Traditional Program

### Traditional program

```text
Input
 ↓
Rules written by programmer
 ↓
Output
```

Example:

```python
if marks >= 50:
    print("Pass")
else:
    print("Fail")
```

### Machine learning model

```text
Training data
 ↓
Learning algorithm
 ↓
Learned parameters
 ↓
Prediction
```

### LLM

```text
Huge text dataset
 ↓
Tokenizer
 ↓
Transformer neural network
 ↓
Large number of parameters
 ↓
Next-token prediction
 ↓
Generated text
```


# 🧪 Student Activity 1 — Change the Dataset

Modify the first dataset so that the model learns about another topic.

For example:

```text
Java is a programming language.
Java is used for software development.
Java supports object oriented programming.
Flutter is used to build applications.
Flutter uses widgets.
```

Then:

1. Re-run the tokenization cells.
2. Re-create the vocabulary.
3. Re-create training examples.
4. Re-create the model.
5. Train it.
6. Test predictions.

### Question

What happens when you use a word that was not present in the training dataset?


# 🧪 Student Activity 2 — Experiment with GPT-2

Try at least five prompts.

Examples:

1. `Explain cloud computing.`
2. `Explain APIs to a beginner.`
3. `Python is`
4. `A good software engineer`
5. `Artificial intelligence can`

Record:

- Prompt
- Generated response
- Was the response useful?
- Was it factually correct?
- What could be improved?


# 🧪 Student Activity 3 — Compare Models

Compare our tiny model with GPT-2.

| Feature | Tiny Model | GPT-2 |
|---|---|---|
| Dataset | Very small | Large |
| Vocabulary | Tiny | Large |
| Architecture | Simple neural network | Transformer |
| Parameters | Very few | Much larger |
| Training | Done in this notebook | Already pretrained |
| Output quality | Very limited | Much better |
| Purpose | Learning | Real model experimentation |

### Discussion

Why can't we train a ChatGPT-sized model in one beginner Colab notebook?


# 🏆 Mini Project — College AI Assistant

Build a simple AI assistant for students.

### Suggested features

The chatbot should answer questions about:

- Programming
- Software engineering
- APIs
- Databases
- Cloud computing
- Artificial intelligence

### Basic flow

```text
Student
   ↓
Question
   ↓
Python Application
   ↓
Language Model
   ↓
Generated Answer
   ↓
Student
```

### Extension

Students can later convert the notebook into:

```text
Web UI
   ↓
Flask / FastAPI
   ↓
LLM
```

or:

```text
React / Flutter
      ↓
   Backend API
      ↓
      LLM
```


# 🔐 Responsible AI

LLMs can generate incorrect information.

Students should understand:

- AI output is not automatically true.
- Do not blindly trust generated answers.
- Do not enter passwords or confidential information.
- Check important facts using reliable sources.
- Respect copyright and privacy.
- Evaluate bias and harmful content.
- Understand the limitations of the model.

### Simple rule

> **Use AI as an assistant, not as an unquestioned authority.**


# 🎓 Final Knowledge Check

Answer these questions before finishing the practical.

### Q1
What does LLM stand for?

### Q2
What is a token?

### Q3
Why do we convert text into numbers?

### Q4
What is a parameter?

### Q5
What is training?

### Q6
What is inference?

### Q7
What is the purpose of a loss function?

### Q8
What is a pretrained model?

### Q9
What is fine-tuning?

### Q10
Why is our tiny language model not comparable to ChatGPT?

### Q11
What is the main idea behind next-token prediction?

### Q12
What is one limitation of LLM-generated answers?


# 🎯 A–Z Summary

```text
A — AI
B — Bias
C — Context
D — Dataset
E — Embedding
F — Fine-tuning
G — Generation
H — Hugging Face
I — Inference
J — JSON / structured data
K — Knowledge
L — LLM
M — Model
N — Neural Network
O — Output
P — Prompt
Q — Query
R — Response
S — Transformer / Sampling
T — Token
U — User input
V — Vocabulary
W — Weights
X — eXperiment
Y — Your dataset
Z — Zero-shot prompting
```

## Final mental model

```text
              LLM
               │
       ┌───────┴────────┐
       │                │
   Training          Inference
       │                │
 Dataset              Prompt
       │                │
 Tokenization       Tokenization
       │                │
 Transformer         Transformer
       │                │
 Parameters       Next-token prediction
       │                │
 Learned model     Generated response
```

### The most important sentence

> **An LLM learns statistical patterns from large amounts of text and uses those learned patterns to predict and generate sequences of tokens.**
